### Check if TensorFlow sees the GPU (optional)

In [4]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

TF version: 2.16.1
GPUs: []


In [1]:
# Basic imports
import os
import numpy as np
import pandas as pd
import re
from tqdm import tqdm

In [6]:
# from datasets import load_dataset

# Load full dataset
# dataset = load_dataset("shenasa/English-Persian-Parallel-Dataset", split="train")

# Save to CSV (may take a few minutes)
# dataset.to_csv("data/processed/full_dataset.csv", index=False)

In [7]:
import pandas as pd

# Replace with your actual path if different
df = pd.read_csv("data/processed/full_dataset.csv")

# Preview
df.head()

,flash fire .,فلاش آتش .
0,superheats the air . burns the lungs like rice...,هوا را فوق العاده گرم می کند . ریه ها را مثل ک...
1,"hey , guys . down here . down here .",سلام بچه ها . این پایین . این پایین .
2,what do you got down this corridor is the bow ...,چه چیزی در این راهرو پایین آمده است ، درست است .
3,theres an access hatch right there that puts u...,یک دریچه دسترسی درست در آنجا وجود دارد که ما ر...
4,we get into the propeller tubes and the only t...,وارد لوله های پروانه می شویم و تنها چیزی که بی...


In [8]:
# 1. Extract the first row (currently column headers)
first_row = df.columns.tolist()

# 2. Insert the first row as actual data
df.loc[-1] = first_row  # insert at index -1 (before index 0)
df.index = df.index + 1  # fix the index
df = df.sort_index()     # sort so new row appears at top

# 3. Rename the columns
df.columns = ['en', 'fa']  # or 'english', 'persian'

# Now df looks correct!
df.head()

,en,fa
0,flash fire .,فلاش آتش .
1,superheats the air . burns the lungs like rice...,هوا را فوق العاده گرم می کند . ریه ها را مثل ک...
2,"hey , guys . down here . down here .",سلام بچه ها . این پایین . این پایین .
3,what do you got down this corridor is the bow ...,چه چیزی در این راهرو پایین آمده است ، درست است .
4,theres an access hatch right there that puts u...,یک دریچه دسترسی درست در آنجا وجود دارد که ما ر...


### Drop Empty Rows

In [9]:
df.dropna(subset=["en", "fa"], inplace=True)
df = df[(df["en"].str.strip() != "") & (df["fa"].str.strip() != "")]

### Filter Very Short / Very Long Sentence Pairs

In [10]:
def is_valid_pair(en, fa, min_len=3, max_len=100):
    en_words = en.split()
    fa_words = fa.split()
    return (
        min_len <= len(en_words) <= max_len
        and min_len <= len(fa_words) <= max_len
        and 0.5 <= len(en_words) / len(fa_words) <= 2
    )

df = df[df.apply(lambda row: is_valid_pair(row["en"], row["fa"]), axis=1)]

Backup before restarting jupyter

In [13]:
df.to_csv("data/processed/cleaned_before_changing_pythonV.csv", index=False)

In [1]:
import pandas as pd
df=pd.read_csv('data\processed\cleaned_before_changing_pythonV.csv')

### Normalize Persian Text (Very Important)

📌 Note: Environment Requirement

⚠️ This step requires Python 3.11, because the hazm library and its dependencies (like fasttext-wheel) are not compatible with Python 3.12+.

To run this cell:

🔄 Switch to the .venv311 virtual environment created with Python 3.11:

.\.venv311\Scripts\activate


Run the normalization code

Then deactivate and switch back to your main environment:

deactivate
.\.venv\Scripts\activate


✅ The output will be a cleaned dataset:
data/processed/cleaned_normalized.csv

In [2]:
from hazm import Normalizer

normalizer = Normalizer()

df["fa"] = df["fa"].apply(lambda x: normalizer.normalize(str(x)))

### Save Cleaned File

In [3]:
df.to_csv("data/processed/cleaned.csv", index=False)

### Reading the file again

In [8]:
df=pd.read_csv(r"data\processed\cleaned.csv")

### Prepare training text

In [9]:
with open("data/processed/cleaned.txt", "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write(row["en"].strip() + "\n")
        f.write(row["fa"].strip() + "\n")

### Train SentencePiece model

In [11]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input="data/processed/cleaned.txt",
    model_prefix="data/tokenizer/spm",
    vocab_size=8000,
    character_coverage=1.0,
    model_type="unigram"  # or "bpe"
)

## Testing  tokenizer

In [12]:
sp = spm.SentencePieceProcessor()
sp.load("data/tokenizer/spm.model")

# Test on English
print(sp.encode("I live in Tehran.", out_type=str))

# Test on Persian
print(sp.encode("من در تهران زندگی می‌کنم.", out_type=str))

['▁I', '▁live', '▁in', '▁Te', 'h', 'ran', '.']
['▁من', '▁در', '▁ت', 'ه', 'ران', '▁زندگی', '▁می', '▁کنم', '.']


In [13]:
sample = "من تو را دوست دارم"
ids = sp.encode(sample, out_type=int)
tokens = sp.encode(sample, out_type=str)

print("Token IDs:", ids)
print("Subword Tokens:", tokens)


Token IDs: [57, 164, 16, 594, 1020]
Subword Tokens: ['▁من', '▁تو', '▁را', '▁دوست', '▁دارم']


In [14]:
df["en_ids"] = df["en"].apply(lambda x: sp.encode(str(x), out_type=int))
df["fa_ids"] = df["fa"].apply(lambda x: sp.encode(str(x), out_type=int))

In [15]:
import pickle

with open("data/processed/tokenized.pkl", "wb") as f:
    pickle.dump(df[["en_ids", "fa_ids"]], f)